# Experiment 4 — XGBoost Focal Loss + Threshold Optimization

**Focal Loss:** Down-weights easy background examples, focuses on hard signal examples.
Originally from RetinaNet (Lin et al. 2017). gamma=2.0 is standard value.

**FIXED from original:** Now includes threshold scan on validation set.
Focal loss improves AUC/Signal Significance but needs lower threshold for F1.

Uses `xgb.train()` API with DMatrix (required for custom objectives).

**Prerequisite:** Run `00_dataset_prep.ipynb` first.

In [1]:
import os, sys, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

XGB_PARAMS = {'max_depth': 6, 'eta': 0.1, 'verbosity': 0, 'seed': 42}
NUM_ROUNDS  = 500
THRESHOLDS  = np.arange(0.1, 0.91, 0.01)
print("Experiment 4 — XGBoost Focal Loss + Threshold Optimization")

Experiment 4 — XGBoost Focal Loss + Threshold Optimization


In [2]:
def focal_loss_gradient(y_pred, dtrain, gamma=2.0):
    """
    Custom focal loss for XGBoost.
    FL = -(1-p_t)^gamma * log(p_t)
    Down-weights easy examples (high confidence), focuses on hard ones.
    gamma=2.0 from Lin et al. 2017 (RetinaNet).

    Returns gradient and hessian (hessian approximated as p*(1-p)).
    """
    y_true = dtrain.get_label()
    p      = 1.0 / (1.0 + np.exp(-y_pred))
    grad   = -(y_true*(1-p)**gamma*(1-p) - (1-y_true)*p**gamma*p)
    hess   = p * (1 - p)
    return grad, hess

In [3]:
all_results = []

for v in ['A', 'B', 'C']:
    print(f"\n[Exp4-XGB] Training Version {v}...")
    try:
        train = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_train.csv'))
        test  = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_test.csv'))
    except FileNotFoundError:
        print(f"ERROR: Run 00_dataset_prep.ipynb first."); raise

    X_train_full = train.drop('label',axis=1).values; y_train_full = train['label'].values
    X_test       = test.drop('label',axis=1).values;  y_test       = test['label'].values

    # Split train into train+validation for threshold tuning
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)

    dtrain = xgb.DMatrix(X_tr,  label=y_tr)
    dval   = xgb.DMatrix(X_val, label=y_val)
    dtest  = xgb.DMatrix(X_test, label=y_test)

    t0 = time.time()
    model = xgb.train(
        XGB_PARAMS, dtrain, num_boost_round=NUM_ROUNDS,
        obj=lambda pred, dtrain: focal_loss_gradient(pred, dtrain, gamma=2.0),
        verbose_eval=False
    )
    train_time = time.time() - t0

    # Scan thresholds on validation set to find best F1
    val_raw   = model.predict(dval)
    val_probs = 1.0 / (1.0 + np.exp(-val_raw))
    f1_scores = [f1_score(y_val, (val_probs>=t).astype(int), pos_label=1, zero_division=0)
                 for t in THRESHOLDS]
    best_threshold = THRESHOLDS[int(np.argmax(f1_scores))]
    print(f"[Exp4-XGB] Version {v}: best_threshold={best_threshold:.2f} | val_F1={max(f1_scores):.4f}")

    # Apply best threshold to test set
    raw_scores = model.predict(dtest)
    probs      = 1.0 / (1.0 + np.exp(-raw_scores))
    preds      = (probs >= best_threshold).astype(int)

    metrics = compute_all_metrics(y_test, probs, preds, train_time)
    metrics['Version']        = v
    metrics['Best_Threshold'] = round(best_threshold, 2)
    all_results.append(metrics)

    np.save(os.path.join(RESULTS_DIR, f'probs_exp4_xgb_version_{v}.npy'), probs)
    print_metrics(metrics, label=f"Exp4-XGB Version {v}")

print("\n[Exp4-XGB] Complete.")


[Exp4-XGB] Training Version A...
[Exp4-XGB] Version A: best_threshold=0.46 | val_F1=0.7684
[Exp4-XGB Version A] AUC=0.8184 | F1=0.7678 | Signal_Sig=170.9354 | Train_Time=563.46s

[Exp4-XGB] Training Version B...
[Exp4-XGB] Version B: best_threshold=0.38 | val_F1=0.3943
[Exp4-XGB Version B] AUC=0.8112 | F1=0.4037 | Signal_Sig=26.9012 | Train_Time=602.65s

[Exp4-XGB] Training Version C...
[Exp4-XGB] Version C: best_threshold=0.30 | val_F1=0.1897
[Exp4-XGB Version C] AUC=0.7816 | F1=0.1872 | Signal_Sig=5.8263 | Train_Time=482.17s

[Exp4-XGB] Complete.


In [4]:
results_df = pd.DataFrame(all_results)[['Version','AUC','F1','Signal_Significance','Train_Time_sec','Best_Threshold']]
save_path  = os.path.join(RESULTS_DIR, 'experiment4_focal_loss_xgb.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")

Version      AUC       F1  Signal_Significance  Train_Time_sec  Best_Threshold
      A 0.818397 0.767759           170.935379          563.46            0.46
      B 0.811250 0.403688            26.901177          602.65            0.38
      C 0.781571 0.187196             5.826287          482.17            0.30

✓ Saved → ..\results\experiment4_focal_loss_xgb.csv
